In [ ]:
from pathlib import Path
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import re
from tqdm.auto import tqdm
import warnings

root = Path("datasets/GSE122662/")

files = sorted(root.glob("GSM319*.h5"))

### Load and save

In [ ]:
# ------------------------------------------------------------
# Parse sample metadata from filename
# ------------------------------------------------------------

def parse_filename(f):
    """
    Expected examples such as:

    GSM3195684_D8.25_serum_C1_gene_bc_mat.h5
    GSM3195685_D8.25_2i_C2_gene_bc_mat.h5
    ...
    """

    name = f.name

    gsm_match = re.search(r"(GSM\d+)", name)
    day_match = re.search(r"_D([\d.]+)", name)
    cond_match = re.search(r"_(serum|2i|Dox)_", name, flags=re.I)
    rep_match = re.search(r"_C(\d+)", name)

    return {
        "sample": gsm_match.group(1) if gsm_match else f.stem,
        "day": float(day_match.group(1)) if day_match else np.nan,
        "condition": cond_match.group(1) if cond_match else "unknown",
        "replicate": rep_match.group(1) if rep_match else "unknown",
    }


# ------------------------------------------------------------
# Process each 10X library
# ------------------------------------------------------------
datasets = []

for i, f in enumerate(files):

    print(f"[{i+1}/{len(files)}] {f.name}")

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Variable names are not unique"
        )
        x = sc.read_10x_h5(f)

    # Original symbols
    x.var["gene_symbol"] = x.var_names.astype(str)

    # Unique Ensembl identifiers
    x.var_names = x.var["gene_ids"].astype(str)

    if not x.var_names.is_unique:
        raise ValueError(f"Gene IDs are not unique in {f}")

    x.X = x.X.astype(np.int32)

    sc.pp.downsample_counts(
        x,
        counts_per_cell=15_000,
        random_state=0,
        replace=False,
    )

    sc.pp.filter_cells(
        x,
        min_counts=2_000,
    )

    meta = parse_filename(f)

    for key, value in meta.items():
        x.obs[key] = value

    x.obs_names = [
        f"{meta['sample']}_{bc}"
        for bc in x.obs_names
    ]

    datasets.append(x)

# ------------------------------------------------------------
# Concatenate all libraries
# ------------------------------------------------------------

adata = ad.concat(
    datasets,
    axis=0,
    join="inner",
    merge="same",
)

print("\nCombined:")
print(adata)


# ------------------------------------------------------------
# Gene filter
#
# Keep genes expressed in >=50 cells ACROSS THE WHOLE DATASET.
# This must therefore happen after concatenation.
# ------------------------------------------------------------

sc.pp.filter_genes(
    adata,
    min_cells=50,
)

print("\nAfter gene filtering:")
print(adata.shape)


# ------------------------------------------------------------
# Save downsampled raw counts
# ------------------------------------------------------------

adata.layers["counts"] = adata.X.copy()


# ------------------------------------------------------------
# Normalize to 10,000 counts / cell
# ------------------------------------------------------------

sc.pp.normalize_total(
    adata,
    target_sum=10_000,
)


# ------------------------------------------------------------
# log(1 + x)
# ------------------------------------------------------------

sc.pp.log1p(adata)


# ------------------------------------------------------------
# Some useful checks
# ------------------------------------------------------------

print("\nFinal object:")
print(adata)

print("\nConditions:")
print(adata.obs["condition"].value_counts())

print("\nDays:")
print(
    adata.obs["day"]
    .value_counts()
    .sort_index()
)

print("\nReplicates:")
print(
    adata.obs["replicate"]
    .value_counts()
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

adata.write_h5ad(
    "datasets/wot_preprocessed_all_genes.h5ad",
    compression="gzip",
)

### Load

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("datasets/wot_preprocessed_all_genes.h5ad")

In [ ]:
import numpy as np

sc.pp.highly_variable_genes(
    adata,
    flavor="seurat",
    n_bins=20,
    min_disp=1.0,
    min_mean=0,
    max_mean=np.inf,
    max_disp=np.inf,
)

print(adata.var["highly_variable"].sum())

# Select HVGs
adata_hvg = adata[:, adata.var["highly_variable"]].copy()

# Go back to raw counts
adata_hvg.X = adata_hvg.layers["counts"].copy()

# Critical: clear inherited log1p flag
adata_hvg.uns.pop("log1p", None)

# Renormalize selected genes
sc.pp.normalize_total(
    adata_hvg,
    target_sum=10_000,
)

sc.pp.log1p(adata_hvg)

In [11]:
days = sorted(adata_hvg.obs["day"].dropna().unique())
print(days)

[np.float64(0.0), np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0), np.float64(5.5), np.float64(6.0), np.float64(6.5), np.float64(7.0), np.float64(7.5), np.float64(8.0), np.float64(8.25), np.float64(8.5), np.float64(8.75), np.float64(9.0), np.float64(9.5), np.float64(10.0), np.float64(10.5), np.float64(11.0), np.float64(11.5), np.float64(12.0), np.float64(12.5), np.float64(13.0), np.float64(13.5), np.float64(14.0), np.float64(14.5), np.float64(15.0), np.float64(15.5), np.float64(16.0), np.float64(16.5), np.float64(17.0), np.float64(17.5), np.float64(18.0)]


In [15]:
daily = {
    day.item(): adata[adata.obs["day"] == day].copy()
    for day in sorted(adata.obs["day"].dropna().unique())
}

In [16]:
daily

{0.0: AnnData object with n_obs × n_vars = 4613 × 19050
     obs: 'n_counts', 'sample', 'day', 'condition', 'replicate'
     var: 'gene_ids', 'gene_symbol', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
     uns: 'log1p', 'hvg'
     layers: 'counts', None,
 0.5: AnnData object with n_obs × n_vars = 3449 × 19050
     obs: 'n_counts', 'sample', 'day', 'condition', 'replicate'
     var: 'gene_ids', 'gene_symbol', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
     uns: 'log1p', 'hvg'
     layers: 'counts', None,
 1.0: AnnData object with n_obs × n_vars = 3679 × 19050
     obs: 'n_counts', 'sample', 'day', 'condition', 'replicate'
     var: 'gene_ids', 'gene_symbol', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
     uns: 'log1p', 'hvg'
     layers: 'counts', None,
 1.5: AnnData object with n_obs × n_vars = 1956 × 19050
     obs: 'n_counts', 'sample', 'day', 'condition', 'replicate'
     var: 'gene_ids', 'gene_